[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/59_paged_kv_cache_solution.ipynb)

# 🔴 Solution: Paged KV Cache

Reference solution for `paged_kv_cache`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

class PagedKVCache:
    def __init__(self, num_layers: int, num_heads: int, head_dim: int,
                 block_size: int, max_blocks: int, device=None, dtype=torch.float32):
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.block_size = block_size
        self.max_blocks = max_blocks
        self.k_blocks = torch.zeros(num_layers, max_blocks, block_size, num_heads, head_dim, device=device, dtype=dtype)
        self.v_blocks = torch.zeros_like(self.k_blocks)
        self.free_blocks = list(range(max_blocks - 1, -1, -1))
        self.tables = {}
        self.lengths = {}

    def _alloc_block(self) -> int:
        if not self.free_blocks:
            raise RuntimeError("PagedKVCache is out of blocks")
        return self.free_blocks.pop()

    def append(self, layer_id: int, seq_id: str, key: torch.Tensor, value: torch.Tensor):
        if key.shape != value.shape:
            raise ValueError("key and value must have the same shape")
        if key.shape[1:] != (self.num_heads, self.head_dim):
            raise ValueError("key/value shape should be (T, num_heads, head_dim)")
        table = self.tables.setdefault(seq_id, [])
        length = self.lengths.get(seq_id, 0)
        offset = 0
        while offset < key.shape[0]:
            block_offset = length % self.block_size
            if block_offset == 0:
                table.append(self._alloc_block())
            block_id = table[-1]
            take = min(self.block_size - block_offset, key.shape[0] - offset)
            self.k_blocks[layer_id, block_id, block_offset:block_offset + take] = key[offset:offset + take]
            self.v_blocks[layer_id, block_id, block_offset:block_offset + take] = value[offset:offset + take]
            offset += take
            length += take
        self.lengths[seq_id] = length

    def get(self, layer_id: int, seq_id: str):
        if seq_id not in self.tables:
            empty = self.k_blocks.new_empty(0, self.num_heads, self.head_dim)
            return empty, empty.clone()
        length = self.lengths[seq_id]
        keys, values = [], []
        remaining = length
        for block_id in self.tables[seq_id]:
            take = min(self.block_size, remaining)
            keys.append(self.k_blocks[layer_id, block_id, :take])
            values.append(self.v_blocks[layer_id, block_id, :take])
            remaining -= take
            if remaining == 0:
                break
        return torch.cat(keys, dim=0), torch.cat(values, dim=0)

    def free(self, seq_id: str):
        blocks = self.tables.pop(seq_id, [])
        self.lengths.pop(seq_id, None)
        self.free_blocks.extend(reversed(blocks))


In [ ]:
# Verify
cache = PagedKVCache(num_layers=1, num_heads=2, head_dim=4, block_size=3, max_blocks=4)
k = torch.randn(5, 2, 4)
v = torch.randn(5, 2, 4)
cache.append(0, "req1", k, v)
print([t.shape for t in cache.get(0, "req1")])


In [ ]:
# Run judge
from torch_judge import check
check('paged_kv_cache')
